In [107]:
import requests
import time
import random
import pandas as pd

# Nivell 1
Des d’un Jupyter Notebook faràs els següents exercicis utilitzant la llibreria requests de Python.
Exploració bàsica amb una API de laboratori.

### 1. Consulta l’API pública JSONPlaceholder utilitzant el mètode GET per obtenir:

*    Llista de publicacions (/posts)
*    Llista d’usuaris (/users)
*    Llista de tasques (/todos)


In [81]:
def request_url_method( url, method='get', **kargs ):
    
    answer = None
    try:
        match method:
            case 'get':
                    answer = requests.get( url )
            case 'post':
                    answer = requests.post( url, json=kargs['json'] )
            case 'patch':
                    answer = requests.patch( url, json=kargs['json'] )
            case 'delete':
                    answer = requests.delete( url )
        
        # No saturar el servidor amb peticions rápides
        wait = 1 + random.uniform( 0, 2 )
        time.sleep( wait )

    except requests.exceptions.RequestException as e:
        print(f" Algo a passat: {e}")
    
    return answer

In [42]:
request_list    = ['posts','users','todos']
request_url     = 'https://jsonplaceholder.typicode.com/'

data_url = {}
for item in request_list:
    data_url[item] = request_url_method( request_url + "/" + item, 'get' )

### 2. Mostra per pantalla:

*    La quantitat total de cada recurs.
*    El codi d'estat de cada petició.


In [105]:
def print_frame_json( json, frame ):

    for line in json[frame['key_data']]:
        for field in frame['key_field']:
            print(f"{field}: {line[field]}")
        print( "-"*60 )

    return None
    

In [100]:
def print_url_answer( answer, frame={} ):
    
    print(f"El seu codi de estat es: {answer.status_code} ")

    match answer.status_code:
        case 200:
            if 'json' in  answer.headers['Content-Type']:
                if not frame:
                    print(f"\t- Hi ha un total de: {len(answer.json())} ")
                    print(f"Contingut: \n{answer.json()}")
                else:
                    print_frame_json( answer.json(), frame )
            else:
                print(f"\t- De moment només se comptar respostes tipus json ... ")
        case 201:
            print(answer.json())
        case 404:
            print("Aquesta no es la pàgina que hi estas buscant ... ")   
    
    return None     

In [44]:
for item, answer in data_url.items():
    print( f"Pel recurs: {item}" )
    print_url_answer( answer )

Pel recurs: posts
El seu codi de estat es: 200 
	- Hi ha un total de: 100 
Pel recurs: users
El seu codi de estat es: 200 
	- Hi ha un total de: 10 
Pel recurs: todos
El seu codi de estat es: 200 
	- Hi ha un total de: 200 


### 3. Fes una petició a una publicació inexistent per obtenir un error 404 i mostra el codi d'estat rebut.

In [79]:
recurs_inex = '/fake'
data_inex_url   = request_url_method( request_url + recurs_inex, 'get' )

In [80]:
print(f"Pel recurs: {recurs_inex}")
print_url_answer( data_inex_url )

Pel recurs: /fake
El seu codi de estat es: 404 
Aquesta no es la pàgina que hi estas buscant ... 


### 4. Fes una petició POST per crear una nova publicació fictícia. Inclou el títol, el cos del missatge i un userId.

*   Mostra la resposta JSON.
*   Mostra el codi d'estat.


In [ ]:
data_post = {
    'userId': 1,
    'title' : 'Lorem ipsum',
    'body'  : 'Lorem ipsum sip inicuanim'
}

url_post = request_url + '/posts'
answer_post = request_url_method( url_post, method='post', json=data_post ) 

In [65]:
print_url_answer( answer_post )

El seu codi de estat es: 201 
{'userId': '1', 'title': 'Lorem ipsum', 'body': 'Lorem ipsum sip inicuanim', 'id': 101}


### 5. Fes una petició PATCH per modificar parcialment una publicació existent.

*   Mostra la resposta JSON.
*   Mostra el codi d'estat.


In [77]:
data_patch={
  'title' : 'ipsum lorem'  
}

url_patch = answer_post.headers['location']
answer_patch = request_url_method( url_patch, method='patch', json=data_patch )

In [78]:
print_url_answer( answer_patch )

El seu codi de estat es: 200 
	- Hi ha un total de: 1 
Contingut: 
{'title': 'ipsum lorem'}


### 6. Fes una petició DELETE sobre una publicació.

*   Mostra la resposta JSON.
*   Mostra el codi d'estat.


In [82]:
url_delete      = answer_post.headers['location']
answer_delete   = request_url_method( url_delete, method='delete' )

In [83]:
print_url_answer( answer_delete )

El seu codi de estat es: 200 
	- Hi ha un total de: 0 
Contingut: 
{}


# Nivell 2
Interacció amb una API pública real

### 1. Explora el repositori de Public APIs i tria una API que permeti fer peticions GET.

API escollida "Art Institute of Chicago" "https://api.artic.edu"

### 2. Llegeix la documentació de l'API:

*   Revisa la secció d’endpoints disponibles. En un markdown, apunta com a mínim dos endpoints diferents.
*   Comprova si ofereix filtres o paràmetres opcionals interessants. En un markdown, anota’ls.
*   Verifica que la resposta sigui en format JSON (és el més adequat per a aquest exercici).


##### EndPoints

*   artworks
*   artists
*   galleries
*   exhibitions
*   artwork-types
*   ...


###### Filtres i parametres:
Generals:
*   `page`      Nombre de la pàgina
*   `limit`     Nombre de recursos per pàgina
*   `fields`    Camps a retornar separats per comes  
*   `ids`       Llista de id's separats per comes

Per artworks:
*   include - Options: artist_pivots, dates, place_pivots, sites

Per .../search:
*   q       - Texte a buscar
*   sort    - Ordenació
*   size    - Nombre de resultats



Els resultats son format Json (per exemple):
```
{
    "data": {
        "id": 4,
        "api_model": "artworks",
        "api_link": "https://api.artic.edu/api/v1/artworks/4",
        "is_boosted": false,
        "title": "Priest and Boy",
        "alt_titles": null,
        ...
    },
    "info": {
        "license_text": "The `description` field in this response is licensed under a Creative Commons Attribution 4.0 Generic License (CC-By) and the Terms and Conditions of artic.edu. All other data in this response is licensed under a Creative Commons Zero (CC0) 1.0 designation and the Terms and Conditions of artic.edu.",
        "license_links": [
            "https://creativecommons.org/publicdomain/zero/1.0/",
            "https://www.artic.edu/terms"
        ],
        "version": "1.14"
    },
    "config": {
        "iiif_url": "https://www-test.artic.edu/iiif/2",
        "website_url": "https://www-test.artic.edu"
    }
}
```

### 3. Fes una petició GET senzilla:

*   Mostra el codi d'estat de la resposta.
*   Imprimeix de forma clara alguns dels camps de la resposta JSON.


In [88]:
API_LINK    = 'https://api.artic.edu/api/v1/'
end_point   = 'artwork-types'
params      = '?limit=15'
request_url = API_LINK + end_point + params
answer_api  = request_url_method( request_url )


In [106]:
frame_api = { 'key_data':'data', 'key_field': ['title', 'api_link'] }
print_url_answer( answer_api, frame= frame_api )

El seu codi de estat es: 200 
title: TBM Equipment
api_link: https://api.artic.edu/api/v1/artwork-types/49
------------------------------------------------------------
title: Painting
api_link: https://api.artic.edu/api/v1/artwork-types/1
------------------------------------------------------------
title: Vessel
api_link: https://api.artic.edu/api/v1/artwork-types/23
------------------------------------------------------------
title: Basketry
api_link: https://api.artic.edu/api/v1/artwork-types/22
------------------------------------------------------------
title: Miniature room
api_link: https://api.artic.edu/api/v1/artwork-types/21
------------------------------------------------------------
title: Model
api_link: https://api.artic.edu/api/v1/artwork-types/20
------------------------------------------------------------
title: Architectural fragment
api_link: https://api.artic.edu/api/v1/artwork-types/19
------------------------------------------------------------
title: Print
api_lin

### 4. Converteix les dades a un DataFrame de pandas:

*   Mostra les primeres files del DataFrame.


In [113]:
df_artwork_types = pd.DataFrame( answer_api.json()[frame_api['key_data']] )
df_artwork_types.head()

,id,api_model,api_link,title,aat_id,source_updated_at,updated_at,timestamp
0,49,artwork-types,https://api.artic.edu/api/v1/artwork-types/49,TBM Equipment,NaN,2026-03-18T16:23:57-05:00,2026-03-18T16:25:22-05:00,2026-06-02T07:58:35-05:00
1,1,artwork-types,https://api.artic.edu/api/v1/artwork-types/1,Painting,300033618.0,2019-05-08T19:03:58-05:00,2022-04-22T10:38:03-05:00,2026-06-02T07:58:35-05:00
2,23,artwork-types,https://api.artic.edu/api/v1/artwork-types/23,Vessel,300193015.0,2019-05-08T19:03:58-05:00,2022-03-15T16:31:40-05:00,2026-06-02T07:58:35-05:00
3,22,artwork-types,https://api.artic.edu/api/v1/artwork-types/22,Basketry,300404578.0,2019-05-08T19:03:58-05:00,2022-03-15T16:31:40-05:00,2026-06-02T07:58:35-05:00
4,21,artwork-types,https://api.artic.edu/api/v1/artwork-types/21,Miniature room,300427608.0,2019-05-08T19:03:59-05:00,2022-03-15T16:31:40-05:00,2026-06-02T07:58:35-05:00


# Nivell 3
API d'Open Data Barcelona

### 1. Utilitza l’API d’Open Data BCN per:

*   Cercar un dataset del teu interès mitjançant package_search i package_show. .


### 2. Dels resultats obtinguts, selecciona’n un que tingui recursos disponibles en CSV o JSON.

### 3. Amb el resource_id del recurs seleccionat, realitza una consulta mitjançant datastore_search per:

*   Recuperar almenys 100 registres.


### 4. Converteix els resultats a un DataFrame.

### 5. Desa el DataFrame en un fitxer .csv.